# 认知拓展 3/8：因果图：混杂、中介与碰撞点

核心问题：**为什么控制更多变量有时反而让结论更错？**

本课不要求背诵模型名称。你要先写预测，再运行小实验，最后说明证据边界。建议投入 60～120 分钟。

固定动作：

观察 → 明确问题 → 选择模型 → 写出预测 → 检查证据 → 寻找反例 → 更新判断

## 课前预测

1. 不查资料，用两句话回答核心问题。
2. 给自己的答案标一个 0%～100% 置信度。
3. 写出一个会使你改变判断的反例。
4. 预测第一个代码实验的方向；运行后不要删除原预测。


## 一手资料与课程取舍

- [Hernán & Robins：Causal Inference: What If](https://miguelhernan.org/whatifbook)
- [Pearl、Glymour、Jewell：因果推断入门](https://web.cs.ucla.edu/~kaoru/primer-complete-2019.pdf)

课程只提取能通过小实验理解的部分。代码输出是教学例子，不是对现实世界的自动裁决。


## 1. 看到相关，先画变量关系

混杂变量同时影响“处理”和“结果”；中介是处理影响结果的路径；碰撞点是两个原因共同指向的结果。该控制什么取决于因果问题，不是“变量越多越科学”。


In [1]:
import numpy as np
rng = np.random.default_rng(7)
n = 20000
ability = rng.normal(size=n)
training = (ability + rng.normal(scale=0.8, size=n) > 0).astype(int)
score = 70 + 8 * ability + 2 * training + rng.normal(scale=4, size=n)

naive = score[training == 1].mean() - score[training == 0].mean()
bins = np.quantile(ability, [0, .2, .4, .6, .8, 1])
adjusted_parts = []
for low, high in zip(bins[:-1], bins[1:]):
    mask = (ability >= low) & (ability <= high)
    adjusted_parts.append(score[mask & (training == 1)].mean() - score[mask & (training == 0)].mean())
print(f"未调整差异={naive:.2f}，按能力分层后的平均差异={np.mean(adjusted_parts):.2f}")


未调整差异=11.90，按能力分层后的平均差异=3.41


## 2. 控制碰撞点会制造不存在的关系

如果“能力”和“关系资源”都能提高录取率，那么只看已录取者会使两者出现负相关：能力较弱但被录取的人更可能有资源，反之亦然。这不是总体中的因果关系。


In [2]:
rng = np.random.default_rng(11)
n = 50000
ability = rng.normal(size=n)
connections = rng.normal(size=n)
admitted = ability + connections + rng.normal(scale=.5, size=n) > 1.0
all_corr = np.corrcoef(ability, connections)[0, 1]
selected_corr = np.corrcoef(ability[admitted], connections[admitted])[0, 1]
print(f"总体相关={all_corr:.3f}，只看已录取者={selected_corr:.3f}")


总体相关=-0.002，只看已录取者=-0.505


## 3. 观察问题与干预问题不同

“使用某功能的人表现更好”是观察关联；“给同类人开启功能会怎样”是干预问题。因果图不能凭数据自动确定，必须结合时间顺序、机制知识和设计。


In [3]:
roles = {
    "设备档次": "可能同时影响是否启用功能和识别率：候选混杂",
    "降噪后 SNR": "若由降噪产生并影响识别：中介",
    "是否进入人工复核": "若由低置信度和高风险共同决定：碰撞点",
}
for variable, role in roles.items():
    print(f"{variable}: {role}")


设备档次: 可能同时影响是否启用功能和识别率：候选混杂
降噪后 SNR: 若由降噪产生并影响识别：中介
是否进入人工复核: 若由低置信度和高风险共同决定：碰撞点


## 误用警报与适用边界

- 工具是对问题的压缩，不是现实本身；先检查假设，再相信输出。
- 一个漂亮数字不能替代数据来源、测量误差、替代解释和失败代价。
- 不要用术语给别人贴标签；把“某某偏差”改写成可检查的判断步骤。
- 不能量化时可以做定性判断，但必须把不知道什么写出来。

## 迁移练习

对同一个工具各写一个例子：

1. ASR/音频：它能防止哪一种误判？
2. 工作：它能改进哪一个会议、指标或项目决策？
3. 日常生活：它能让哪个选择更可逆、更可检验？

每个例子都要包含：原判断、工具带来的新问题、更新后的行动。


## 闭卷挑战

画一张“使用降噪功能→识别率”的因果图，至少包含一个混杂、一个中介和一个选择变量；分别说明控制它们会回答什么问题。

用下面的判断卡作答：

    问题：
    当前主张：
    概率或置信度：
    关键证据：
    最强替代解释：
    什么结果会让我改主意：
    适用边界：
    下一步最小行动：

## 最小掌握门禁

- [ ] 能不用术语解释本课工具。
- [ ] 能从空白重写至少一个核心函数。
- [ ] 能构造一个让直觉失败的反例。
- [ ] 能指出工具本身的一种误用。
- [ ] 能迁移到 ASR、工作、生活三个场景。
- [ ] 已把一条预测和实际结果写入 LEARNING_LOG.md。
